In [ ]:
#this same notebook was used for probing 3k samples and 4k samples with changes in cell 2 and 3

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
import json
import os

In [33]:
# ── CHANGE THESE TWO LINES per run ──────────────────────────────
MODEL_FOLDER = "muril-base-cased" # "bert-base-multilingual-cased" or "muril-base-cased" or "xlm-roberta-base"
STATE        =  "finetuned"  # "pretrained" or "finetuned"
# ────────────────────────────────────────────────────────────────

BASE_PATH  = "/Users/harshaggarwal/Projects_4/hinemo_project/models"
BASE_DIR   = f"{BASE_PATH}/hidden_states_full_means_4k_othwerwise_3k"
MODEL_DIR  = f"{BASE_DIR}/{MODEL_FOLDER}"
OUTPUT_DIR = f"{BASE_PATH}/phase7a_probing_results_4k_lambda/{MODEL_FOLDER}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Model:  {MODEL_FOLDER}")
print(f"State:  {STATE}")

Model:  muril-base-cased
State:  finetuned


In [34]:
hidden = np.load(f"{MODEL_DIR}/hidden_{STATE}_full.npy")
meta   = pd.read_csv(f"{MODEL_DIR}/metadata_full.csv")

lambda_vals = meta["lambda"].values
n_samples, n_layers, hidden_dim = hidden.shape

print(f"Hidden states shape: {hidden.shape}")
print(f"Lambda values shape: {lambda_vals.shape}")

Hidden states shape: (4000, 12, 768)
Lambda values shape: (4000,)


In [35]:
r2_per_layer = []

for layer in range(n_layers):
    X = hidden[:, layer, :]   # shape: (3000, 768) — representations at this layer
    y = lambda_vals            # shape: (3000,)     — λ values to predict

    probe  = Ridge(alpha=1.0)
    scores = cross_val_score(probe, X, y, cv=5, scoring="r2")
    mean_r2 = scores.mean()

    r2_per_layer.append(mean_r2)
    print(f"Layer {layer+1:2d}: R² = {mean_r2:.4f}")

Layer  1: R² = 0.1929
Layer  2: R² = 0.6904
Layer  3: R² = 0.7235
Layer  4: R² = 0.7169
Layer  5: R² = 0.6993
Layer  6: R² = 0.6816
Layer  7: R² = 0.5155
Layer  8: R² = 0.1969
Layer  9: R² = 0.2175
Layer 10: R² = 0.1127
Layer 11: R² = 0.4655
Layer 12: R² = 0.3217


In [36]:
LSL = int(np.argmax(r2_per_layer)) + 1   # +1 because layers are 1-indexed in the paper

print(f"\nLSL ({MODEL_FOLDER}, {STATE}) = Layer {LSL}")
print(f"Peak R²                       = {r2_per_layer[LSL-1]:.4f}")
print(f"\nFull R² curve:")
for i, r2 in enumerate(r2_per_layer):
    marker = " ← LSL" if i+1 == LSL else ""
    print(f"  Layer {i+1:2d}: {r2:.4f}{marker}")


LSL (muril-base-cased, finetuned) = Layer 3
Peak R²                       = 0.7235

Full R² curve:
  Layer  1: 0.1929
  Layer  2: 0.6904
  Layer  3: 0.7235 ← LSL
  Layer  4: 0.7169
  Layer  5: 0.6993
  Layer  6: 0.6816
  Layer  7: 0.5155
  Layer  8: 0.1969
  Layer  9: 0.2175
  Layer 10: 0.1127
  Layer 11: 0.4655
  Layer 12: 0.3217


In [37]:
results = {
    "model"       : MODEL_FOLDER,
    "state"       : STATE,
    "r2_per_layer": r2_per_layer,
    "LSL"         : LSL,
    "peak_r2"     : r2_per_layer[LSL-1],
}

save_path = f"{OUTPUT_DIR}/probing_{STATE}.json"
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {save_path}")

Results saved to /Users/harshaggarwal/Projects_4/hinemo_project/models/phase7a_probing_results_4k_lambda/muril-base-cased/probing_finetuned.json


In [38]:
#summary script
import json
import os

BASE_PATH = "/Users/harshaggarwal/Projects_4/hinemo_project/models"

models = [
    "bert-base-multilingual-cased",
    "muril-base-cased", 
    "xlm-roberta-base"
]
states = ["pretrained", "finetuned"]

print(f"{'Model':<35} {'State':<12} {'LSL':<6} {'Peak R²'}")
print("-" * 65)

for model in models:
    for state in states:
        path = f"{BASE_PATH}/phase7a_probing_results_4k_lambda/{model}/probing_{state}.json"
        if os.path.exists(path):
            with open(path) as f:
                r = json.load(f)
            print(f"{model:<35} {state:<12} {r['LSL']:<6} {r['peak_r2']:.4f}")
        else:
            print(f"{model:<35} {state:<12} {'--':<6} --  (not yet run)")

Model                               State        LSL    Peak R²
-----------------------------------------------------------------
bert-base-multilingual-cased        pretrained   5      0.7171
bert-base-multilingual-cased        finetuned    4      0.7050
muril-base-cased                    pretrained   4      0.7584
muril-base-cased                    finetuned    3      0.7235
xlm-roberta-base                    pretrained   12     0.7051
xlm-roberta-base                    finetuned    1      0.6562
